In [ ]:
%load_ext autoreload
%autoreload 2

# Genetic Algroithm

In [ ]:
import os
import pickle
import sys
import pandas as pd
import colour
from dotenv import load_dotenv
from Code.Utils.util_methods import UtilMethods
from Code.GA.utils import add_other_columns, normalize_individual
from tqdm.notebook import tqdm
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from Code.GA.utils import recipe_to_lab
from skimage.color import lab2rgb
import matplotlib.pyplot as plt
import shutil
from Code.GA.GA_scripts import run_ga
import numpy as np
from Code.GA.utils import summarize_results
from Code.GA.utils import metrics_counter





base = UtilMethods.find_project_root(os.getcwd())
print(f"Project root found: {base}")

if load_dotenv(f'{base}/.env'):
    print(".env found")
else:
    print("ERROR .env not found")

In [ ]:
REFLECTANCE = True 
LIGHT_SOURCE = 'F2' # F2 or D65 or StudioLED
INPUT_PATH = f'{base}/Dataset/traintest/X.csv'

## Load the data

In [ ]:
# model

with open(f'{base}/Dataset/GP_pipeline_models_dict.pkl', 'rb') as f:
    pipeline_dict_file = pickle.load(f)

if REFLECTANCE:
    model =  pipeline_dict_file['model']['curve']
elif LIGHT_SOURCE == 'F2':
    model = pipeline_dict_file['model']['Lab']
elif LIGHT_SOURCE == 'D65':
    model = pipeline_dict_file['model']['LabD65']
elif LIGHT_SOURCE == 'StudioLED':
    model = pipeline_dict_file['model']['LabStudioLED']
else:
    raise Exception

In [ ]:
full_recipes_df = pd.read_pickle(f'{base}/Dataset/unique_recipes.pkl')
pigments_df = pd.read_csv(f'{base}/Dataset/traintest/y.csv')
max_pigment_values = pigments_df.max()

## Run the GA

In [ ]:
step = 10

for i in tqdm(range(0, len(pigments_df)), desc=f'Iterating through all recipes by steps of {step}'):

    if i % step != 0:
        continue

    row = pigments_df.iloc[[i]]

    count_per_row = (row > 0.0).sum(axis=1).iloc[0]
    if count_per_row <= 1:
        continue

    save_folder = f'{base}/Code/GA/Results/Validation-Metamerism/recipe_{i}'

    if os.path.exists(save_folder):
        if os.path.exists(f'{save_folder}/.done'):
            continue
        else:
            # delte the folder and recreate later
            shutil.rmtree(save_folder)



    for j in range(3):
        run_ga(X.iloc[[i]], model, max_pigment_values, occurences, pipeline_dict_file, all_pigments, population=None, generations=200, population_size=300,
                                    tournament_size=30, mutation_rate=0.5, min_mutation=0.4, max_mutation=1.6, zero_prob=0.05,
                                    reflectance=REFLECTANCE, early_stopping=5, early_stopping_start=3, early_stopping_tolerance=0.0005,
                                    mandatory_pigments=None, forbidden_pigments=None, expected_recipe=row, uncertanity_bias=0.05,
                                    plot_fitness=False, save_results=True, save_final_result=True, debug=False, plot_best_colors=False, 
                                    save_folder=save_folder, visualization_offset=50, progressbar_text=f'Running genetic algroithm for recipe {i} - {j+1}')
    
    with open(f'{save_folder}/.done', 'w') as f:
        f.write(f"{datetime.now(ZoneInfo('Europe/Amsterdam')).strftime('%Y-%m-%dT%H:%M:%SZ')}\n")
        f.write('INTERNAL FLAG TO INDICATE THAT THE GA HAS COMPLETED FOR THIS RECIPE!\nDO NOT DELETE OR MODIFY!!!!')






In [ ]:
summarize_results(f'{base}/Code/GA/Results/Validation-Metamerism', modulo=10)

## Creation of plots to analyze the performance

In [ ]:
non_zero_counts = (pigments_df != 0).sum().sort_values(ascending=True)
non_zero_counts = non_zero_counts[non_zero_counts > 0].sort_values(ascending=True)


# create horizontal bar plot
plt.figure(figsize=(10, len(pigments_df.columns) * 0.1))  # make the plot tall
plt.barh(y=non_zero_counts.index, width=non_zero_counts.values)
plt.xlabel('total number of occurence')
plt.ylabel('pigment')
plt.title('Total amount off occurence of pigments in the historical recipes')

# add count labels at the end of each bar
for i, (col, val) in enumerate(non_zero_counts.items()):
    plt.text(val + 0.5, i, str(val), va='center')  # 0.5 offset for spacing
plt.tight_layout()
plt.show()

In [ ]:
metrics = metrics_counter(f'{base}/Code/GA/Results/Validation-Metamerism/summary.csv')
metrics

In [ ]:
labels = [f'{str(i).zfill(2)}-{str(i+1).zfill(2)}' for i in range(10)]
values = [metrics[f'fitness{i}'] for i in labels]

fig, ax = plt.subplots()
bars = ax.bar(labels, values)

total = metrics['total_values']

# add padding to the y-axis
max_height = max(values)
ax.set_ylim(0, max_height * 1.10)  # increase upper limit

# show values above bars
for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,   # x position
        height,                              # y position
        f'{height}\n{((height/total)*100):.2f}%',                         # text
        ha='center', va='bottom'             # horizontal and vertical alignment
    )

plt.title('Count of fitness values per groups')
plt.xlabel('Group of fitness *10')
plt.ylabel('Count')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# sample data
labels = ['0-1', '1-2', '2-4', '4+']
x = np.arange(4)
input1 = [metrics[f'dE94_{i}'] for i in labels]   # values for left y-axis
input2 = [metrics[f'dE94_{i}_pigments_mean'] for i in labels]  # values for right y-axis
input2_err = [metrics[f'dE94_{i}_pigments_std'] for i in labels]  # std dev for red bars

# bar width
width = 0.4

fig, ax1 = plt.subplots()

# add padding to the y-axis
max_height = max(input1)
ax1.set_ylim(0, max_height * 1.10)  # increase upper limit

# plot input1 on ax1
bars1 = ax1.bar(x - width/2, input1, width, label='Input 1', color='tab:blue')
ax1.set_ylabel('Size of each group', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

# annotate input1 bars
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, height + 1, f'{height:.0f}\n{((height/total)*100):.2f}%',
             ha='center', va='bottom', fontsize=8, color='tab:blue')

# create a second y-axis
ax2 = ax1.twinx()

# plot input2 on ax2 with error bars
bars2 = ax2.bar(
    x + width/2, input2, width,
    yerr=input2_err, capsize=5,
    label='Input 2', color='tab:red',
    error_kw={'linewidth': 0.5, 'capthick': 0.5}  # error bars
)
ax2.set_ylabel('Average amount of pigments in each group', color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')

# annotate input2 bars with ± std
for bar, err in zip(bars2, input2_err):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, height + 0.01 * height,
             f'{height:.2f}\n±{err:.2f}', ha='center', va='bottom',
             fontsize=8, color='tab:red')

# x-axis settings
plt.xticks(x, labels)
ax1.set_xlabel('dE94 groups')

# title
plt.title('dE94 color distance groups and average pigment count')

# layout
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# sample data
labels = ['0-1', '1-2', '2-4', '4+']
x = np.arange(len(labels))

# blue bars: size of each group
input1 = [metrics[f'dE94_{i}'] for i in labels]  # left y-axis

# red bars: mean fitness values
fitness_means = [metrics[f'dE94_{i}_fitness_mean'] for i in labels]  # right y-axis
fitness_stds = [metrics[f'dE94_{i}_fitness_std'] for i in labels]

# bar width
width = 0.4

# figure and axes
fig, ax1 = plt.subplots()

# add padding to the y-axis
max_height = max(input1)
ax1.set_ylim(0, max_height * 1.10)  # increase upper limit

# plot blue bars (group sizes) on ax1
bars1 = ax1.bar(x - width/2, input1, width, color='tab:blue')
ax1.set_ylabel('Size of each group', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

# annotate blue bars
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, height + 1, f'{height:.0f}\n{((height/total)*100):.2f}%',
             ha='center', va='bottom', fontsize=8, color='tab:blue')

# create second y-axis
ax2 = ax1.twinx()

# plot red bars (fitness means) on ax2 with std as error bars
bars2 = ax2.bar(
    x + width/2, fitness_means, width,
    yerr=fitness_stds, capsize=5,
    color='tab:red',
    error_kw={'linewidth': 0.5, 'capthick': 0.5}
)
ax2.set_ylabel('Mean fitness', color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')

# annotate red bars with mean ± std
for bar, err in zip(bars2, fitness_stds):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, height + 0.01 * height,
             f'{height:.2f}\n±{err:.2f}', ha='center', va='bottom',
             fontsize=8, color='tab:red')

# x-axis settings
plt.xticks(x, labels)
ax1.set_xlabel('dE94 groups')

# title
plt.title('dE94 color distance groups and mean fitness values')

# layout
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# sample data
labels = ['0-1', '1-2', '2-4', '4+']
x = np.arange(len(labels))

# blue bars: size of each group
input1 = [metrics[f'dE94_{i}'] for i in labels]  # left y-axis

# red bars: mean fitness values
fitness_means = [metrics[f'dE94_{i}_dEm_mean'] for i in labels]  # right y-axis
fitness_stds = [metrics[f'dE94_{i}_dEm_std'] for i in labels]

# bar width
width = 0.4

# figure and axes
fig, ax1 = plt.subplots()

# add padding to the y-axis
max_height = max(input1)
ax1.set_ylim(0, max_height * 1.10)  # increase upper limit

# plot blue bars (group sizes) on ax1
bars1 = ax1.bar(x - width/2, input1, width, color='tab:blue')
ax1.set_ylabel('Size of each group', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

# annotate blue bars
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, height + 1, f'{height:.0f}\n{((height/total)*100):.2f}%',
             ha='center', va='bottom', fontsize=8, color='tab:blue')

# create second y-axis
ax2 = ax1.twinx()

# plot red bars (fitness means) on ax2 with std as error bars
bars2 = ax2.bar(
    x + width/2, fitness_means, width,
    yerr=fitness_stds, capsize=5,
    color='tab:red',
    error_kw={'linewidth': 0.5, 'capthick': 0.5}
)
ax2.set_ylabel('Mean dEm', color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')

# annotate red bars with mean ± std
for bar, err in zip(bars2, fitness_stds):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, height + 0.01 * height,
             f'{height:.2f}\n±{err:.2f}', ha='center', va='bottom',
             fontsize=8, color='tab:red')

# x-axis settings
plt.xticks(x, labels)
ax1.set_xlabel('dE94 groups')

# title
plt.title('dE94 color distance groups and mean dEm values')

# layout
plt.tight_layout()
plt.show()


In [ ]:
save_folder = f'{base}/Code/GA/Results/Validation/Test'

In [ ]:
# run_ga(expectation_reflectance, model, max_pigment_values, occurences, pipeline_dict_file, population=None, generations=200, population_size=300,
#                           tournament_size=30, mutation_rate=0.5, min_mutation=0.4, max_mutation=1.6, zero_prob=0.05,
#                           reflectance=REFLECTANCE, early_stopping=5, early_stopping_start=3, early_stopping_tolerance=0.0005,
#                           mandatory_pigments=None, forbidden_pigments=None, expected_recipe=expectation_recipe, uncertanity_bias=0.05,
#                           plot_fitness=False, save_results=True, save_final_result=True, debug=False, plot_best_colors=False, save_folder = save_folder, visualization_offset=50)
